# exp_ensemble_diagnose — why did the R ensemble land at 0.52 (the old 7-feature level)?

Reloads the cached features (no GPU — cross-encoder scores are in `ce_feats_R.npz`) and rebuilds the
exact train/test matrices, then runs: (a) single-feature NDCG, (b) feature correlations, (c) leave-one-out,
(d) num_leaves sweep, (e) judge score range vs floor. Tests: redundancy on R / num_leaves mis-pick / floor.


## Setup (Colab — CPU is fine)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q lightgbm pytrec_eval datasets pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd, lightgbm as lgb
from datasets import load_dataset
from ctmatch.experiments import ExperimentConfig, load_eval, ndcg_at_k
cfg = ExperimentConfig(data_root=DATA_ROOT)
FEATURES = ['bm25','bm25_rank','dense','dense_rank','rrf','clf_rel','clf_partial','v2_rel','llm_yesno']
RANK_FEATS = {'bm25_rank','dense_rank'}   # lower = better
TRAIN_SETS, TEST_SETS = ['trec21','kz'], ['trec22']


In [ ]:
# Reload everything and rebuild the exact matrices (identical to train_ensemble_full.build()).
corpus_ids = {r['text'].strip() for r in load_dataset(cfg.index2docid_hf, data_files='index2docid.txt', split='train')}
sets = load_eval(cfg, TRAIN_SETS + TEST_SETS)
pool = json.load(open(cfg.path('data/pool_R.json')))
rfeat = {}
for l in open(cfg.path('data/retrieval_feats_R.jsonl')):
    r = json.loads(l); rfeat[(r['source'], r['topic_id'], r['doc_id'])] = r
llm = {}
for l in open(cfg.path('data/llm_scores_R.jsonl')):
    r = json.loads(l); llm[(r['source'], r['topic_id'], r['doc_id'])] = r['llm_score']
z = np.load(cfg.path('cache/ce_feats_R.npz'), allow_pickle=True); clf_f, v2_f = z['clf'].item(), z['v2'].item()
def build(setnames):
    X, y, groups, meta = [], [], [], []
    for s in setnames:
        rel = sets[s]['rel_dict']
        for t, docs in pool[s].items():
            docs = [d for d in docs if d in corpus_ids]; groups.append(len(docs))
            for d in docs:
                rf = rfeat.get((s,t,d), {}); cr, cp = clf_f.get((s,t,d),(0.,0.)); v2 = v2_f.get((s,t,d),(0.,0.))[0]
                X.append([rf.get('bm25',0.), rf.get('bm25_rank',cfg.cand_k), rf.get('dense',0.), rf.get('dense_rank',cfg.cand_k),
                          rf.get('rrf',0.), cr, cp, v2, llm.get((s,t,d), cfg.llm_floor)])
                y.append(int(rel[t].get(d,0))); meta.append((s,t,d))
    return np.array(X,dtype=np.float32), np.array(y), groups, meta
Xtr,ytr,gtr,_ = build(TRAIN_SETS); Xte,yte,gte,mte = build(TEST_SETS)
print('train', Xtr.shape, 'test', Xte.shape)


In [ ]:
# Helper: NDCG@10 on TREC22 from a score vector aligned to Xte rows.
def te_ndcg(scores):
    vals, i0 = [], 0
    for g in gte:
        yy = yte[i0:i0+g]; ss = scores[i0:i0+g]
        vals.append(ndcg_at_k([d for d,_ in sorted(zip(range(g), ss), key=lambda x:-x[1])],
                              {i: int(yy[i]) for i in range(g)})); i0 += g
    return float(np.mean(vals))
def train_eval(cols, num_leaves=31, min_data=20, rounds=50):
    b = lgb.train({'objective':'lambdarank','metric':'ndcg','ndcg_eval_at':[10],'num_leaves':num_leaves,
                   'min_data_in_leaf':min_data,'learning_rate':0.05,'lambda_l2':1.0,'verbose':-1},
                  lgb.Dataset(Xtr[:,cols], ytr, group=gtr), num_boost_round=rounds)
    return te_ndcg(b.predict(Xte[:,cols]))


### (a) Single-feature NDCG@10 on TREC22 — which features are individually strong?


In [ ]:
rows = []
for j, f in enumerate(FEATURES):
    s = -Xte[:, j] if f in RANK_FEATS else Xte[:, j]   # rank feats: lower is better
    rows.append({'feature': f, 'ndcg@10': round(te_ndcg(s), 4)})
print('full 9-feature ensemble (num_leaves=31):', round(train_eval(list(range(9))), 4))
pd.DataFrame(rows).sort_values('ndcg@10', ascending=False)


### (b) Feature correlations — are the eligibility readers (clf_rel/v2_rel/llm_yesno) redundant?


In [ ]:
C = np.corrcoef(Xte.T)
pd.DataFrame(C.round(2), index=FEATURES, columns=FEATURES)


### (c) Leave-one-out — does dropping v2_rel / llm_yesno actually change anything?


In [ ]:
base = train_eval(list(range(9)))
loo = [{'dropped': '(none)', 'ndcg@10': round(base,4), 'delta': 0.0}]
for j, f in enumerate(FEATURES):
    cols = [k for k in range(9) if k != j]; v = train_eval(cols)
    loo.append({'dropped': f, 'ndcg@10': round(v,4), 'delta': round(v-base,4)})
pd.DataFrame(loo)


### (d) num_leaves sweep — did CV mis-pick 31 over 15?


In [ ]:
pd.DataFrame([{'num_leaves': nl, 'ndcg@10': round(train_eval(list(range(9)), num_leaves=nl), 4)} for nl in [7,15,31,63]])


### (e) Judge score range vs floor — is llm_floor=-36 inside the new judge's range?


In [ ]:
js = np.array([v for v in llm.values()])
print(f'judge score: min={js.min():.2f} max={js.max():.2f} mean={js.mean():.2f} | llm_floor={cfg.llm_floor}')
print('floor is', 'INSIDE the range (miscalibrated!)' if js.min() < cfg.llm_floor else 'below the range (OK)')


## Reading it
- **(a)** if clf_rel/v2_rel/llm_yesno each score high alone but **(c)** dropping them barely moves NDCG, and **(b)** they're highly correlated (>0.7) → **redundancy on R** (hypothesis #1): they carry the same signal, so the ensemble can't stack them. Fix = an *orthogonal* topicality feature (§9e #2), not tuning.
- **(d)** if num_leaves=15 clearly beats 31 → CV mis-picked (KZ noise); refit with 15.
- **(e)** if min < -36 → floor bug is back; set the floor below the true min.
